In [1]:
"""
================================================================================
QATAR AIRWAYS — KEYWORD FREQUENCY TABLE BUILDER
Splits pipe-separated NLP keyword columns into Power BI-ready tables
Senior Data Specialist Portfolio — Product Development & Design Team
================================================================================

INPUT:
    qatar_sentiment_flat.csv   ← output from NLP pipeline

OUTPUTS:
    qatar_keywords_positive.csv   ← one row per keyword, positive sentiment
    qatar_keywords_negative.csv   ← one row per keyword, negative sentiment
    qatar_keywords_combined.csv   ← both combined with sentiment column
    qatar_top_complaints.csv      ← frequency table of top_complaint field
    qatar_top_praises.csv         ← frequency table of top_praise field
    qatar_theme_summary.csv       ← product theme negativity rates for heatmap

POWER BI USAGE:
    Import all CSVs as separate tables.
    Join qatar_keywords_*.csv to sentiment flat on review_id for
    filtered word frequency charts by cabin, year, route etc.
================================================================================
"""

import os
import pandas as pd
import numpy as np
from collections import Counter

# ── CONFIG ────────────────────────────────────────────────────────────────────
INPUT_FILE  = "qatar_sentiment_flat.csv"   # adjust path if needed
OUTPUT_DIR  = "./"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── LOAD ──────────────────────────────────────────────────────────────────────
print("Loading sentiment flat file...")
df = pd.read_csv(INPUT_FILE)
print(f"  {len(df):,} reviews loaded")
print(f"  Columns: {list(df.columns)}")
print()

# Normalise date and year
df["review_date"] = pd.to_datetime(df["review_date"], errors="coerce")
df["year"]  = pd.to_numeric(df["year"],  errors="coerce")
df["month"] = pd.to_numeric(df["month"], errors="coerce")

# ── HELPER ────────────────────────────────────────────────────────────────────
def explode_keywords(df, col, sentiment_label):
    """
    Splits pipe-separated keyword strings into one row per keyword.
    Carries through all review metadata for Power BI cross-filtering.
    """
    rows = []
    for _, row in df.iterrows():
        raw = row.get(col, "")
        if pd.isna(raw) or str(raw).strip() == "":
            continue
        keywords = [k.strip().lower() for k in str(raw).split("|") if k.strip()]
        for kw in keywords:
            rows.append({
                "review_id":         row.get("review_id"),
                "review_date":       row.get("review_date"),
                "year":              row.get("year"),
                "month":             row.get("month"),
                "seat_type":         row.get("seat_type"),
                "type_of_traveller": row.get("type_of_traveller"),
                "route":             row.get("route"),
                "reviewer_country":  row.get("reviewer_country"),
                "overall_rating_10": row.get("overall_rating_10"),
                "sentiment_overall": row.get("sentiment_overall"),
                "sentiment_score":   row.get("sentiment_score"),
                "keyword":           kw,
                "sentiment_type":    sentiment_label,
            })
    return pd.DataFrame(rows)

# ── 1. POSITIVE KEYWORDS ──────────────────────────────────────────────────────
print("Building positive keywords table...")
pos_df = explode_keywords(df, "positive_keywords", "positive")

# Add frequency rank globally and per cabin
pos_freq = pos_df["keyword"].value_counts().reset_index()
pos_freq.columns = ["keyword", "frequency"]
pos_freq["rank_global"] = pos_freq["frequency"].rank(
    ascending=False, method="dense").astype(int)
pos_df = pos_df.merge(pos_freq[["keyword","frequency","rank_global"]], on="keyword")

pos_path = os.path.join(OUTPUT_DIR, "qatar_keywords_positive.csv")
pos_df.to_csv(pos_path, index=False)
print(f"  ✓ {len(pos_df):,} rows → {pos_path}")
print(f"  Top 10 positive keywords:")
print(pos_freq.head(10).to_string(index=False))
print()

# ── 2. NEGATIVE KEYWORDS ──────────────────────────────────────────────────────
print("Building negative keywords table...")
neg_df = explode_keywords(df, "negative_keywords", "negative")

neg_freq = neg_df["keyword"].value_counts().reset_index()
neg_freq.columns = ["keyword", "frequency"]
neg_freq["rank_global"] = neg_freq["frequency"].rank(
    ascending=False, method="dense").astype(int)
neg_df = neg_df.merge(neg_freq[["keyword","frequency","rank_global"]], on="keyword")

neg_path = os.path.join(OUTPUT_DIR, "qatar_keywords_negative.csv")
neg_df.to_csv(neg_path, index=False)
print(f"  ✓ {len(neg_df):,} rows → {neg_path}")
print(f"  Top 10 negative keywords:")
print(neg_freq.head(10).to_string(index=False))
print()

# ── 3. COMBINED KEYWORDS TABLE ────────────────────────────────────────────────
print("Building combined keywords table...")
combined_df = pd.concat([pos_df, neg_df], ignore_index=True)

# Recalculate global frequency within each sentiment type
combined_df["freq_within_type"] = combined_df.groupby(
    ["sentiment_type","keyword"])["keyword"].transform("count")

combined_path = os.path.join(OUTPUT_DIR, "qatar_keywords_combined.csv")
combined_df.to_csv(combined_path, index=False)
print(f"  ✓ {len(combined_df):,} rows → {combined_path}")
print()

# ── 4. TOP COMPLAINTS FREQUENCY TABLE ────────────────────────────────────────
print("Building top complaints table...")
complaints = df["top_complaint"].dropna()
complaints = complaints[complaints.str.strip() != ""]
complaints = complaints[complaints.str.lower() != "null"]
complaints = complaints[complaints.str.lower() != "none"]

complaint_freq = (complaints
    .str.lower()
    .str.strip()
    .value_counts()
    .reset_index())
complaint_freq.columns = ["complaint", "frequency"]
complaint_freq["rank"] = complaint_freq["frequency"].rank(
    ascending=False, method="dense").astype(int)

# Also build with metadata for cross-filtering
complaint_rows = []
for _, row in df.iterrows():
    c = row.get("top_complaint", "")
    if pd.isna(c) or str(c).strip() in ["", "null", "None"]:
        continue
    complaint_rows.append({
        "review_id":         row.get("review_id"),
        "year":              row.get("year"),
        "seat_type":         row.get("seat_type"),
        "type_of_traveller": row.get("type_of_traveller"),
        "route":             row.get("route"),
        "overall_rating_10": row.get("overall_rating_10"),
        "sentiment_score":   row.get("sentiment_score"),
        "top_complaint":     str(c).lower().strip(),
    })
complaint_detail = pd.DataFrame(complaint_rows)
complaint_detail = complaint_detail.merge(
    complaint_freq[["complaint","frequency","rank"]],
    left_on="top_complaint", right_on="complaint", how="left"
).drop(columns=["complaint"])

complaints_path = os.path.join(OUTPUT_DIR, "qatar_top_complaints.csv")
complaint_detail.to_csv(complaints_path, index=False)
print(f"  ✓ {len(complaint_detail):,} rows → {complaints_path}")
print(f"  Top 15 complaints:")
print(complaint_freq.head(15).to_string(index=False))
print()

# ── 5. TOP PRAISES FREQUENCY TABLE ───────────────────────────────────────────
print("Building top praises table...")
praises = df["top_praise"].dropna()
praises = praises[praises.str.strip() != ""]
praises = praises[praises.str.lower() != "null"]
praises = praises[praises.str.lower() != "none"]

praise_freq = (praises
    .str.lower()
    .str.strip()
    .value_counts()
    .reset_index())
praise_freq.columns = ["praise", "frequency"]
praise_freq["rank"] = praise_freq["frequency"].rank(
    ascending=False, method="dense").astype(int)

praise_rows = []
for _, row in df.iterrows():
    p = row.get("top_praise", "")
    if pd.isna(p) or str(p).strip() in ["", "null", "None"]:
        continue
    praise_rows.append({
        "review_id":         row.get("review_id"),
        "year":              row.get("year"),
        "seat_type":         row.get("seat_type"),
        "type_of_traveller": row.get("type_of_traveller"),
        "route":             row.get("route"),
        "overall_rating_10": row.get("overall_rating_10"),
        "sentiment_score":   row.get("sentiment_score"),
        "top_praise":        str(p).lower().strip(),
    })
praise_detail = pd.DataFrame(praise_rows)
praise_detail = praise_detail.merge(
    praise_freq[["praise","frequency","rank"]],
    left_on="top_praise", right_on="praise", how="left"
).drop(columns=["praise"])

praises_path = os.path.join(OUTPUT_DIR, "qatar_top_praises.csv")
praise_detail.to_csv(praises_path, index=False)
print(f"  ✓ {len(praise_detail):,} rows → {praises_path}")
print(f"  Top 15 praises:")
print(praise_freq.head(15).to_string(index=False))
print()

# ── 6. PRODUCT THEME SUMMARY TABLE ───────────────────────────────────────────
print("Building product theme summary table...")

theme_cols = [c for c in df.columns if c.startswith("theme_")]
theme_labels = {
    "theme_seat_comfort":  "Seat Comfort",
    "theme_cabin_crew":    "Cabin Crew",
    "theme_food_beverage": "Food & Beverage",
    "theme_ife":           "IFE",
    "theme_wifi":          "Wi-Fi",
    "theme_lounge":        "Lounge",
    "theme_boarding":      "Boarding",
    "theme_cleanliness":   "Cleanliness",
    "theme_value":         "Value for Money",
    "theme_punctuality":   "Punctuality",
}

theme_rows = []
for col in theme_cols:
    if col not in df.columns:
        continue
    label = theme_labels.get(col, col)
    for cabin in df["seat_type"].dropna().unique():
        sub = df[df["seat_type"] == cabin]
        mentioned = sub[sub[col] != "not_mentioned"]
        if len(mentioned) == 0:
            continue
        n_pos  = (mentioned[col] == "positive").sum()
        n_neg  = (mentioned[col] == "negative").sum()
        n_neut = (mentioned[col] == "neutral").sum()
        n_total = len(mentioned)
        theme_rows.append({
            "theme":           label,
            "theme_col":       col,
            "seat_type":       cabin,
            "n_mentioned":     n_total,
            "n_positive":      n_pos,
            "n_negative":      n_neg,
            "n_neutral":       n_neut,
            "pct_positive":    round(n_pos  / n_total * 100, 1),
            "pct_negative":    round(n_neg  / n_total * 100, 1),
            "pct_neutral":     round(n_neut / n_total * 100, 1),
            "sentiment_ratio": round((n_pos - n_neg) / n_total, 3),
        })

    # Also add overall (all cabins)
    mentioned_all = df[df[col] != "not_mentioned"]
    if len(mentioned_all) > 0:
        n_pos  = (mentioned_all[col] == "positive").sum()
        n_neg  = (mentioned_all[col] == "negative").sum()
        n_neut = (mentioned_all[col] == "neutral").sum()
        n_total = len(mentioned_all)
        theme_rows.append({
            "theme":           label,
            "theme_col":       col,
            "seat_type":       "All Cabins",
            "n_mentioned":     n_total,
            "n_positive":      n_pos,
            "n_negative":      n_neg,
            "n_neutral":       n_neut,
            "pct_positive":    round(n_pos  / n_total * 100, 1),
            "pct_negative":    round(n_neg  / n_total * 100, 1),
            "pct_neutral":     round(n_neut / n_total * 100, 1),
            "sentiment_ratio": round((n_pos - n_neg) / n_total, 3),
        })

theme_df = pd.DataFrame(theme_rows)
theme_path = os.path.join(OUTPUT_DIR, "qatar_theme_summary.csv")
theme_df.to_csv(theme_path, index=False)
print(f"  ✓ {len(theme_df):,} rows → {theme_path}")
print()
print("  Theme negativity rates (All Cabins):")
all_cabin_themes = theme_df[theme_df["seat_type"] == "All Cabins"][
    ["theme","pct_negative","pct_positive","n_mentioned"]
].sort_values("pct_negative", ascending=False)
print(all_cabin_themes.to_string(index=False))
print()

# ── 7. SENTIMENT TREND TABLE (monthly aggregates for Power BI line charts) ───
print("Building monthly sentiment trend table...")

monthly_rows = []
for (year, month), grp in df.groupby(["year","month"]):
    if pd.isna(year) or pd.isna(month):
        continue
    monthly_rows.append({
        "year":                  int(year),
        "month":                 int(month),
        "date":                  f"{int(year)}-{int(month):02d}-01",
        "n_reviews":             len(grp),
        "avg_sentiment_score":   round(grp["sentiment_score"].mean(), 4),
        "avg_overall_rating":    round(pd.to_numeric(grp["overall_rating_10"],
                                        errors="coerce").mean(), 3),
        "pct_positive":          round((grp["sentiment_overall"]=="positive").mean()*100, 1),
        "pct_negative":          round((grp["sentiment_overall"]=="negative").mean()*100, 1),
        "pct_neutral":           round((grp["sentiment_overall"]=="neutral").mean()*100, 1),
        "pct_mixed":             round((grp["sentiment_overall"]=="mixed").mean()*100, 1),
        "pct_recommended":       round((grp["recommended"]=="Yes").mean()*100, 1)
                                 if "recommended" in grp.columns else None,
        "pct_sarcastic":         round(grp["is_sarcastic"].astype(bool).mean()*100, 1)
                                 if "is_sarcastic" in grp.columns else None,
        "pct_competitor_mention":round(grp["competitor_mentioned"].astype(bool).mean()*100, 1)
                                 if "competitor_mentioned" in grp.columns else None,
    })

monthly_df = pd.DataFrame(monthly_rows).sort_values(["year","month"])
monthly_path = os.path.join(OUTPUT_DIR, "qatar_sentiment_monthly.csv")
monthly_df.to_csv(monthly_path, index=False)
print(f"  ✓ {len(monthly_df):,} rows → {monthly_path}")
print()

# ── SUMMARY ───────────────────────────────────────────────────────────────────
print("=" * 65)
print("ALL TABLES BUILT — READY FOR POWER BI")
print("=" * 65)
print()
print("Import these files into Power BI:")
print()
files = [
    ("qatar_sentiment_flat.csv",     "Core NLP table — one row per review"),
    ("qatar_keywords_positive.csv",  "Positive keyword frequency — for bar/word charts"),
    ("qatar_keywords_negative.csv",  "Negative keyword frequency — for bar/word charts"),
    ("qatar_keywords_combined.csv",  "Both keywords — use sentiment_type as slicer"),
    ("qatar_top_complaints.csv",     "Top complaints — frequency ranked"),
    ("qatar_top_praises.csv",        "Top praises — frequency ranked"),
    ("qatar_theme_summary.csv",      "Theme negativity rates — for heatmap matrix"),
    ("qatar_sentiment_monthly.csv",  "Monthly aggregates — for trend lines"),
]
for fname, desc in files:
    print(f"  {fname:<40} {desc}")

print()
print("Power BI relationships:")
print("  sentiment_flat[review_id]      → reviews[review_id]          (1:1)")
print("  keywords_positive[review_id]   → sentiment_flat[review_id]   (many:1)")
print("  keywords_negative[review_id]   → sentiment_flat[review_id]   (many:1)")
print("  top_complaints[review_id]      → sentiment_flat[review_id]   (1:1)")
print("  top_praises[review_id]         → sentiment_flat[review_id]   (1:1)")
print("  theme_summary                  → standalone (no join needed)")
print("  sentiment_monthly              → standalone (no join needed)")
print()
print("Recommended visuals per table:")
print("  keywords_combined  → Bar chart: keyword by frequency, filter by sentiment_type")
print("  theme_summary      → Matrix heatmap: theme vs seat_type, values=pct_negative")
print("  sentiment_monthly  → Line chart: avg_sentiment_score + avg_overall_rating")
print("  top_complaints     → Bar chart: top_complaint by frequency, top 15")
print("  top_praises        → Bar chart: top_praise by frequency, top 15")

Loading sentiment flat file...
  1,957 reviews loaded
  Columns: ['review_id', 'review_date', 'year', 'month', 'reviewer_country', 'seat_type', 'type_of_traveller', 'route', 'aircraft', 'overall_rating_10', 'recommended', 'trip_verified', 'review_title', 'score_seat_comfort', 'score_cabin_staff', 'score_food_beverages', 'score_ife', 'score_ground_service', 'score_wifi', 'score_value_for_money', 'sentiment_overall', 'sentiment_score', 'emotion_primary', 'is_sarcastic', 'sarcasm_flag_reason', 'theme_seat_comfort', 'theme_cabin_crew', 'theme_food_beverage', 'theme_ife', 'theme_wifi', 'theme_lounge', 'theme_boarding', 'theme_cleanliness', 'theme_value', 'theme_punctuality', 'top_complaint', 'top_praise', 'competitor_mentioned', 'competitor_name', 'competitor_sentiment', 'would_rebook_signal', 'service_recovery_mentioned', 'loyalty_program_mentioned', 'positive_keywords', 'negative_keywords', 'product_implications']

Building positive keywords table...
  ✓ 6,587 rows → ./qatar_keywords_posi

In [2]:
!pip install sentence-transformers scikit-learn

   ---------------------------------------- 0.0/596.4 kB ? eta -:--:--
   ---------------------------------------- 596.4/596.4 kB 2.7 MB/s  0:00:00
   ---------------------------------------- 0.0/11.2 MB ? eta -:--:--
   ---- ----------------------------------- 1.3/11.2 MB 7.4 MB/s eta 0:00:02
   ----------- ---------------------------- 3.1/11.2 MB 8.2 MB/s eta 0:00:01
   ------------------ --------------------- 5.2/11.2 MB 8.4 MB/s eta 0:00:01
   ------------------------ --------------- 6.8/11.2 MB 8.6 MB/s eta 0:00:01
   ------------------------------- -------- 8.7/11.2 MB 8.6 MB/s eta 0:00:01
   ------------------------------------ --- 10.2/11.2 MB 8.4 MB/s eta 0:00:01
   ---------------------------------------  11.0/11.2 MB 8.4 MB/s eta 0:00:01
   ---------------------------------------  11.0/11.2 MB 8.4 MB/s eta 0:00:01
   ---------------------------------------- 11.2/11.2 MB 6.7 MB/s  0:00:01
   ---------------------------------------- 0.0/721.1 kB ? eta -:--:--
   --------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [3]:
"""
================================================================================
QATAR AIRWAYS — SEMANTIC COMPLAINT & PRAISE CLUSTERING
Sentence Transformers + K-Means | Portfolio Piece — PDD Team
================================================================================

WHAT THIS DOES:
    Groups fragmented complaint/praise text into meaningful semantic clusters.
    "rude staff", "dismissive crew", "unhelpful attendant" → one cluster.
    Each cluster gets an auto-generated product-team-friendly label.

INSTALL:
    pip install sentence-transformers scikit-learn pandas numpy matplotlib
    seaborn umap-learn

    Note: sentence-transformers downloads ~90MB model on first run.
    Model: all-MiniLM-L6-v2 — fast, accurate, no GPU needed.

INPUT:
    qatar_sentiment_flat.csv   ← from NLP pipeline

OUTPUTS:
    qatar_complaints_clustered.csv   ← complaints with cluster labels
    qatar_praises_clustered.csv      ← praises with cluster labels
    qatar_cluster_summary.csv        ← cluster frequency table for Power BI
    fig_clusters_complaints.png      ← UMAP visualisation of complaint clusters
    fig_clusters_praises.png         ← UMAP visualisation of praise clusters
================================================================================
"""

# ── IMPORTS ───────────────────────────────────────────────────────────────────
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import normalize
from collections import Counter

warnings.filterwarnings("ignore")
os.makedirs("charts", exist_ok=True)

# ── CONFIG ────────────────────────────────────────────────────────────────────
INPUT_FILE       = "qatar_sentiment_flat.csv"   # adjust path if needed
EMBEDDING_MODEL  = "all-MiniLM-L6-v2"          # fast, accurate, ~90MB
N_COMPLAINT_CLUSTERS = 8    # tune: 6-10 for this dataset size
N_PRAISE_CLUSTERS    = 7    # tune: 5-9 for this dataset size
RANDOM_STATE     = 42
MIN_TEXT_LENGTH  = 3        # ignore very short entries like "null" or "na"

# Qatar Airways brand palette
C = {
    "maroon":      "#8D1B3D",
    "steel_dark":  "#4A6278",
    "steel_mid":   "#6B8296",
    "grey_dark":   "#5A5A5A",
    "grey_light":  "#C8C9CA",
    "off_white":   "#F7F5F2",
    "positive":    "#2E7D5A",
    "negative":    "#C0392B",
    "warning":     "#C47B00",
}

CLUSTER_PALETTE = [
    "#8D1B3D","#4A6278","#6B8296","#2E7D5A",
    "#C47B00","#5A5A5A","#C0392B","#1A5276",
    "#7D6608","#1E8449",
]

plt.rcParams.update({
    "figure.facecolor":  C["off_white"],
    "axes.facecolor":    C["off_white"],
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.grid":         True,
    "grid.alpha":        0.2,
    "grid.linestyle":    "--",
    "font.family":       "sans-serif",
    "font.size":         11,
})


# ── LOAD DATA ─────────────────────────────────────────────────────────────────
print("Loading sentiment data...")
df = pd.read_csv(INPUT_FILE)
print(f"  {len(df):,} reviews loaded")

def clean_text_series(series):
    """Remove nulls, placeholders, very short strings."""
    cleaned = (series
        .dropna()
        .astype(str)
        .str.strip()
        .str.lower()
    )
    cleaned = cleaned[~cleaned.isin([
        "null","none","n/a","na","not mentioned",
        "not stated","","nan"
    ])]
    cleaned = cleaned[cleaned.str.len() >= MIN_TEXT_LENGTH]
    return cleaned

complaints_raw = clean_text_series(df["top_complaint"])
praises_raw    = clean_text_series(df["top_praise"])

print(f"  Valid complaints : {len(complaints_raw):,}")
print(f"  Valid praises    : {len(praises_raw):,}")
print(f"  Unique complaints: {complaints_raw.nunique():,}")
print(f"  Unique praises   : {praises_raw.nunique():,}")
print()

# Keep index to join back to df metadata later
complaints_df = df.loc[complaints_raw.index, [
    "review_id","year","month","seat_type","type_of_traveller",
    "route","reviewer_country","overall_rating_10","sentiment_score",
    "recommended","top_complaint"
]].copy()
complaints_df["top_complaint_clean"] = complaints_raw

praises_df = df.loc[praises_raw.index, [
    "review_id","year","month","seat_type","type_of_traveller",
    "route","reviewer_country","overall_rating_10","sentiment_score",
    "recommended","top_praise"
]].copy()
praises_df["top_praise_clean"] = praises_raw


# ── LOAD EMBEDDING MODEL ──────────────────────────────────────────────────────
print(f"Loading embedding model: {EMBEDDING_MODEL}")
print("  (Downloads ~90MB on first run — cached after that)")
model = SentenceTransformer(EMBEDDING_MODEL)
print("  Model loaded ✓")
print()


# ── OPTIMAL K FINDER ─────────────────────────────────────────────────────────
def find_optimal_k(embeddings, k_range=range(4, 12), label=""):
    """
    Run silhouette analysis to find optimal number of clusters.
    Silhouette score measures how well each point fits its cluster
    vs adjacent clusters. Higher = better defined clusters.
    """
    print(f"  Finding optimal k for {label} (silhouette analysis)...")
    scores = {}
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=RANDOM_STATE,
                    n_init=10, max_iter=300)
        labels = km.fit_predict(embeddings)
        score  = silhouette_score(embeddings, labels, sample_size=min(1000, len(embeddings)))
        scores[k] = score
        print(f"    k={k}: silhouette={score:.4f}")
    best_k = max(scores, key=scores.get)
    print(f"  → Optimal k={best_k} (score={scores[best_k]:.4f})")
    return best_k, scores


# ── CLUSTER LABELLER ──────────────────────────────────────────────────────────
def generate_cluster_label(texts, sentiment_type="complaint"):
    """
    Generate a product-team-friendly label from the most frequent
    terms in a cluster. No LLM needed — TF-IDF style term extraction.
    """
    from collections import Counter
    import re

    STOPWORDS = {
        "the","a","an","and","or","but","in","on","at","to","for",
        "of","with","is","was","were","are","be","been","being",
        "have","has","had","do","does","did","will","would","could",
        "should","may","might","shall","can","need","must","very",
        "really","quite","also","just","even","still","already",
        "much","more","most","some","any","all","no","not","too",
        "so","as","it","its","my","our","their","this","that","these",
        "those","i","we","they","he","she","you","qatar","airways",
        "flight","airline","passenger","service","qr","staff",
    }

    all_words = []
    for text in texts:
        words = re.findall(r'\b[a-z]{3,}\b', str(text).lower())
        all_words.extend([w for w in words if w not in STOPWORDS])

    freq = Counter(all_words)
    top_terms = [word for word, _ in freq.most_common(3)]

    if not top_terms:
        return "General " + sentiment_type.title()

    # Build label from top terms
    label = " ".join(top_terms).title()
    return label


# ══════════════════════════════════════════════════════════════════════════════
# PART 1 — COMPLAINTS CLUSTERING
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("PART 1 — CLUSTERING COMPLAINTS")
print("=" * 60)

print("Generating complaint embeddings...")
complaint_texts     = complaints_df["top_complaint_clean"].tolist()
complaint_embeddings = model.encode(complaint_texts,
                                     show_progress_bar=True,
                                     batch_size=64)
complaint_embeddings_norm = normalize(complaint_embeddings)
print(f"  Embeddings shape: {complaint_embeddings_norm.shape}")
print()

# Find optimal k
best_k_c, sil_scores_c = find_optimal_k(
    complaint_embeddings_norm,
    k_range=range(4, min(12, len(complaint_texts)//10)),
    label="complaints"
)

# Override with config value if preferred
# best_k_c = N_COMPLAINT_CLUSTERS
print()

# Final clustering
print(f"Clustering complaints into {best_k_c} clusters...")
km_c = KMeans(n_clusters=best_k_c, random_state=RANDOM_STATE,
              n_init=15, max_iter=500)
complaint_labels = km_c.fit_predict(complaint_embeddings_norm)
complaints_df["cluster_id"] = complaint_labels

# Generate cluster labels
print("Generating cluster labels...")
complaint_cluster_info = {}
for cluster_id in range(best_k_c):
    mask  = complaints_df["cluster_id"] == cluster_id
    texts = complaints_df.loc[mask, "top_complaint_clean"].tolist()
    label = generate_cluster_label(texts, "complaint")
    freq  = mask.sum()
    complaint_cluster_info[cluster_id] = {
        "label":      label,
        "frequency":  freq,
        "pct":        round(freq / len(complaints_df) * 100, 1),
        "examples":   texts[:5],
    }
    print(f"  Cluster {cluster_id:2d} ({freq:4d} complaints) → {label}")
    print(f"    Examples: {texts[:3]}")

# Map labels back
complaints_df["cluster_label"] = complaints_df["cluster_id"].map(
    {k: v["label"] for k, v in complaint_cluster_info.items()}
)

# Avg sentiment per cluster
cluster_sentiment = complaints_df.groupby("cluster_label").agg(
    frequency        = ("review_id", "count"),
    avg_rating       = ("overall_rating_10", lambda x: pd.to_numeric(x, errors="coerce").mean()),
    avg_sentiment    = ("sentiment_score", "mean"),
    pct_recommended  = ("recommended", lambda x: (x=="Yes").mean()*100),
).round(3).sort_values("frequency", ascending=False)

print()
print("Complaint cluster summary:")
print(cluster_sentiment.to_string())
print()


# ══════════════════════════════════════════════════════════════════════════════
# PART 2 — PRAISES CLUSTERING
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("PART 2 — CLUSTERING PRAISES")
print("=" * 60)

print("Generating praise embeddings...")
praise_texts      = praises_df["top_praise_clean"].tolist()
praise_embeddings = model.encode(praise_texts,
                                  show_progress_bar=True,
                                  batch_size=64)
praise_embeddings_norm = normalize(praise_embeddings)
print(f"  Embeddings shape: {praise_embeddings_norm.shape}")
print()

best_k_p, sil_scores_p = find_optimal_k(
    praise_embeddings_norm,
    k_range=range(4, min(12, len(praise_texts)//10)),
    label="praises"
)

print()
print(f"Clustering praises into {best_k_p} clusters...")
km_p = KMeans(n_clusters=best_k_p, random_state=RANDOM_STATE,
              n_init=15, max_iter=500)
praise_labels = km_p.fit_predict(praise_embeddings_norm)
praises_df["cluster_id"] = praise_labels

print("Generating cluster labels...")
praise_cluster_info = {}
for cluster_id in range(best_k_p):
    mask  = praises_df["cluster_id"] == cluster_id
    texts = praises_df.loc[mask, "top_praise_clean"].tolist()
    label = generate_cluster_label(texts, "praise")
    freq  = mask.sum()
    praise_cluster_info[cluster_id] = {
        "label":     label,
        "frequency": freq,
        "pct":       round(freq / len(praises_df) * 100, 1),
        "examples":  texts[:5],
    }
    print(f"  Cluster {cluster_id:2d} ({freq:4d} praises) → {label}")
    print(f"    Examples: {texts[:3]}")

praises_df["cluster_label"] = praises_df["cluster_id"].map(
    {k: v["label"] for k, v in praise_cluster_info.items()}
)

praise_sentiment = praises_df.groupby("cluster_label").agg(
    frequency        = ("review_id", "count"),
    avg_rating       = ("overall_rating_10", lambda x: pd.to_numeric(x, errors="coerce").mean()),
    avg_sentiment    = ("sentiment_score", "mean"),
    pct_recommended  = ("recommended", lambda x: (x=="Yes").mean()*100),
).round(3).sort_values("frequency", ascending=False)

print()
print("Praise cluster summary:")
print(praise_sentiment.to_string())
print()


# ══════════════════════════════════════════════════════════════════════════════
# PART 3 — UMAP VISUALISATIONS
# ══════════════════════════════════════════════════════════════════════════════
print("Generating UMAP visualisations...")

try:
    from umap import UMAP

    def plot_umap(embeddings, labels, cluster_info, title, filename, sentiment_type):
        print(f"  Running UMAP for {sentiment_type}...")
        reducer = UMAP(n_components=2, random_state=RANDOM_STATE,
                       n_neighbors=15, min_dist=0.1)
        coords  = reducer.fit_transform(embeddings)

        fig, ax = plt.subplots(figsize=(14, 9))
        fig.suptitle(title, fontsize=15, fontweight="bold",
                     color=C["maroon"], y=1.01)

        unique_clusters = sorted(set(labels))
        legend_patches  = []

        for i, cluster_id in enumerate(unique_clusters):
            mask   = np.array(labels) == cluster_id
            colour = CLUSTER_PALETTE[i % len(CLUSTER_PALETTE)]
            label  = cluster_info[cluster_id]["label"]
            freq   = cluster_info[cluster_id]["frequency"]

            ax.scatter(
                coords[mask, 0], coords[mask, 1],
                s=35, alpha=0.65, color=colour,
                edgecolors="none", zorder=3
            )

            # Cluster centroid label
            cx = coords[mask, 0].mean()
            cy = coords[mask, 1].mean()
            ax.annotate(
                f"{label}\n(n={freq})",
                (cx, cy),
                fontsize=8.5, fontweight="bold", color=colour,
                ha="center", va="center",
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                          edgecolor=colour, alpha=0.85, linewidth=1.5),
                zorder=5,
            )
            legend_patches.append(
                mpatches.Patch(color=colour, label=f"{label} (n={freq})")
            )

        ax.set_title(
            "Each point = one passenger review | Proximity = semantic similarity",
            fontsize=10, color=C["grey_dark"], pad=6
        )
        ax.set_xlabel("UMAP Dimension 1", fontsize=10)
        ax.set_ylabel("UMAP Dimension 2", fontsize=10)
        ax.legend(handles=legend_patches, fontsize=8,
                  loc="lower right", framealpha=0.9)
        ax.set_facecolor(C["off_white"])
        fig.patch.set_facecolor(C["off_white"])

        fig.text(0.99, 0.01,
                 "Qatar Airways | Semantic Clustering: all-MiniLM-L6-v2 + K-Means | PDD Portfolio",
                 ha="right", va="bottom", fontsize=7.5,
                 color=C["grey_dark"], alpha=0.5)

        path = f"charts/{filename}"
        fig.savefig(path, dpi=180, bbox_inches="tight",
                    facecolor=C["off_white"])
        plt.close(fig)
        print(f"  ✓ Saved {path}")

    plot_umap(
        complaint_embeddings_norm, complaint_labels,
        complaint_cluster_info,
        "Complaint Semantic Clusters — Qatar Airways Passenger Reviews",
        "fig_clusters_complaints.png",
        "complaints"
    )

    plot_umap(
        praise_embeddings_norm, praise_labels,
        praise_cluster_info,
        "Praise Semantic Clusters — Qatar Airways Passenger Reviews",
        "fig_clusters_praises.png",
        "praises"
    )

except ImportError:
    print("  umap-learn not installed — skipping UMAP visualisation")
    print("  Install with: pip install umap-learn")
    print("  (Optional — all CSV outputs still generated without it)")


# ══════════════════════════════════════════════════════════════════════════════
# PART 4 — POWER BI CHART: CLUSTER FREQUENCY WITH CABIN BREAKDOWN
# ══════════════════════════════════════════════════════════════════════════════
print("\nGenerating cluster frequency charts...")

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
fig.suptitle(
    "Qatar Airways — Semantic Complaint & Praise Clusters\n"
    "Frequency Analysis with Cabin Class Breakdown",
    fontsize=15, fontweight="bold", y=1.02, color=C["maroon"]
)

# Complaints chart
ax = axes[0]
c_summary = complaints_df.groupby(
    ["cluster_label","seat_type"]).size().unstack(fill_value=0)
c_summary["total"] = c_summary.sum(axis=1)
c_summary = c_summary.sort_values("total", ascending=True)

cabin_cols = {
    "Economy Class":  C["steel_dark"],
    "Business Class": C["maroon"],
    "First Class":    C["steel_mid"],
    "Premium Economy":C["grey_dark"],
}
bottom = np.zeros(len(c_summary))
for cabin in ["Economy Class","Business Class","First Class","Premium Economy"]:
    if cabin in c_summary.columns:
        vals = c_summary[cabin].values
        ax.barh(c_summary.index, vals, left=bottom,
                color=cabin_cols[cabin], edgecolor="none",
                height=0.65, label=cabin)
        bottom += vals

for i, (idx, row) in enumerate(c_summary.iterrows()):
    ax.text(row["total"] + 1, i, str(int(row["total"])),
            va="center", fontsize=9, color=C["grey_dark"])

ax.set_title("Top Complaint Clusters by Cabin", fontweight="bold")
ax.set_xlabel("Number of Reviews")
ax.legend(fontsize=9, loc="lower right")
ax.set_xlim(0, c_summary["total"].max() * 1.18)

# Praises chart
ax = axes[1]
p_summary = praises_df.groupby(
    ["cluster_label","seat_type"]).size().unstack(fill_value=0)
p_summary["total"] = p_summary.sum(axis=1)
p_summary = p_summary.sort_values("total", ascending=True)

bottom = np.zeros(len(p_summary))
for cabin in ["Economy Class","Business Class","First Class","Premium Economy"]:
    if cabin in p_summary.columns:
        vals = p_summary[cabin].values
        ax.barh(p_summary.index, vals, left=bottom,
                color=cabin_cols[cabin], edgecolor="none",
                height=0.65, label=cabin)
        bottom += vals

for i, (idx, row) in enumerate(p_summary.iterrows()):
    ax.text(row["total"] + 1, i, str(int(row["total"])),
            va="center", fontsize=9, color=C["grey_dark"])

ax.set_title("Top Praise Clusters by Cabin", fontweight="bold")
ax.set_xlabel("Number of Reviews")
ax.legend(fontsize=9, loc="lower right")
ax.set_xlim(0, p_summary["total"].max() * 1.18)

fig.text(0.99, 0.01,
         "Qatar Airways | Semantic Clustering: all-MiniLM-L6-v2 + K-Means | PDD Portfolio",
         ha="right", va="bottom", fontsize=7.5,
         color=C["grey_dark"], alpha=0.5)

path = "charts/fig_clusters_frequency.png"
fig.savefig(path, dpi=180, bbox_inches="tight", facecolor=C["off_white"])
plt.close(fig)
print(f"  ✓ Saved {path}")


# ══════════════════════════════════════════════════════════════════════════════
# PART 5 — SAVE POWER BI READY CSVS
# ══════════════════════════════════════════════════════════════════════════════
print("\nSaving Power BI ready CSVs...")

# Complaints with clusters
complaints_out = complaints_df[[
    "review_id","year","month","seat_type","type_of_traveller",
    "route","reviewer_country","overall_rating_10","sentiment_score",
    "recommended","top_complaint","top_complaint_clean",
    "cluster_id","cluster_label",
]].copy()
complaints_out["sentiment_type"] = "complaint"
path = "qatar_complaints_clustered.csv"
complaints_out.to_csv(path, index=False)
print(f"  ✓ {path} ({len(complaints_out):,} rows)")

# Praises with clusters
praises_out = praises_df[[
    "review_id","year","month","seat_type","type_of_traveller",
    "route","reviewer_country","overall_rating_10","sentiment_score",
    "recommended","top_praise","top_praise_clean",
    "cluster_id","cluster_label",
]].copy()
praises_out["sentiment_type"] = "praise"
path = "qatar_praises_clustered.csv"
praises_out.to_csv(path, index=False)
print(f"  ✓ {path} ({len(praises_out):,} rows)")

# Master cluster summary for Power BI heatmap/bar
complaint_summary_rows = []
for cluster_id, info in complaint_cluster_info.items():
    sub = complaints_df[complaints_df["cluster_id"] == cluster_id]
    complaint_summary_rows.append({
        "sentiment_type":    "complaint",
        "cluster_id":        cluster_id,
        "cluster_label":     info["label"],
        "frequency":         info["frequency"],
        "pct_of_total":      info["pct"],
        "avg_rating":        pd.to_numeric(sub["overall_rating_10"],
                             errors="coerce").mean().round(2),
        "avg_sentiment":     sub["sentiment_score"].mean().round(3),
        "pct_recommended":   (sub["recommended"]=="Yes").mean()*100,
        "top_cabin":         sub["seat_type"].value_counts().index[0]
                             if len(sub) > 0 else "",
        "top_route":         sub["route"].value_counts().index[0]
                             if len(sub) > 0 else "",
        "examples":          " | ".join(info["examples"][:3]),
    })

praise_summary_rows = []
for cluster_id, info in praise_cluster_info.items():
    sub = praises_df[praises_df["cluster_id"] == cluster_id]
    praise_summary_rows.append({
        "sentiment_type":    "praise",
        "cluster_id":        cluster_id,
        "cluster_label":     info["label"],
        "frequency":         info["frequency"],
        "pct_of_total":      info["pct"],
        "avg_rating":        pd.to_numeric(sub["overall_rating_10"],
                             errors="coerce").mean().round(2),
        "avg_sentiment":     sub["sentiment_score"].mean().round(3),
        "pct_recommended":   (sub["recommended"]=="Yes").mean()*100,
        "top_cabin":         sub["seat_type"].value_counts().index[0]
                             if len(sub) > 0 else "",
        "top_route":         sub["route"].value_counts().index[0]
                             if len(sub) > 0 else "",
        "examples":          " | ".join(info["examples"][:3]),
    })

cluster_summary = pd.DataFrame(
    complaint_summary_rows + praise_summary_rows
).sort_values(["sentiment_type","frequency"], ascending=[True,False])

path = "qatar_cluster_summary.csv"
cluster_summary.to_csv(path, index=False)
print(f"  ✓ {path} ({len(cluster_summary):,} rows)")

# ── FINAL SUMMARY ─────────────────────────────────────────────────────────────
print()
print("=" * 65)
print("CLUSTERING COMPLETE")
print("=" * 65)
print()
print(f"  Complaint clusters : {best_k_c}")
print(f"  Praise clusters    : {best_k_p}")
print()
print("Outputs:")
print("  qatar_complaints_clustered.csv  ← complaints + cluster labels")
print("  qatar_praises_clustered.csv     ← praises + cluster labels")
print("  qatar_cluster_summary.csv       ← summary table for Power BI")
print("  charts/fig_clusters_complaints.png  ← UMAP semantic map")
print("  charts/fig_clusters_praises.png     ← UMAP semantic map")
print("  charts/fig_clusters_frequency.png   ← frequency by cabin chart")
print()
print("Power BI — import these tables:")
print("  qatar_cluster_summary.csv")
print("    → Horizontal bar: cluster_label by frequency, filter by sentiment_type")
print("    → Matrix: cluster_label × top_cabin, values=frequency")
print("    → Scatter: avg_rating vs pct_recommended, size=frequency")
print()
print("  qatar_complaints_clustered.csv + qatar_praises_clustered.csv")
print("    → Join to sentiment_flat on review_id for full cross-filtering")
print("    → Slicer by cluster_label reveals which routes/cabins drive each cluster")

c:\Users\DanielED\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading sentiment data...
  1,957 reviews loaded
  Valid complaints : 1,183
  Valid praises    : 1,456
  Unique complaints: 1,172
  Unique praises   : 1,444

Loading embedding model: all-MiniLM-L6-v2
  (Downloads ~90MB on first run — cached after that)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9512.36it/s]


  Model loaded ✓

PART 1 — CLUSTERING COMPLAINTS
Generating complaint embeddings...


Batches: 100%|██████████| 19/19 [00:01<00:00, 18.45it/s]


  Embeddings shape: (1183, 384)

  Finding optimal k for complaints (silhouette analysis)...
    k=4: silhouette=0.0734
    k=5: silhouette=0.0809
    k=6: silhouette=0.0783
    k=7: silhouette=0.0830
    k=8: silhouette=0.0915
    k=9: silhouette=0.0864
    k=10: silhouette=0.0958
    k=11: silhouette=0.0820
  → Optimal k=10 (score=0.0958)

Clustering complaints into 10 clusters...
Generating cluster labels...
  Cluster  0 ( 185 complaints) → Poor Customer Ife
    Examples: ['zero communication, no alternatives', 'no compensation offered, nothing.', 'no support, no transport']
  Cluster  1 ( 206 complaints) → Food Poor Quality
    Examples: ['no food or used entertainment', 'sandwich no choice', 'chaotic boarding, wrong food']
  Cluster  2 ( 136 complaints) → Delayed Missed Delay
    Examples: ['return flight cancelled 4 times', 'champagne ran out early, wine ran out mid-flight', 'segments delayed minimum 1 hour']
  Cluster  3 (  67 complaints) → Luggage Baggage Lost
    Examples: ['c

Batches: 100%|██████████| 23/23 [00:01<00:00, 14.45it/s]


  Embeddings shape: (1456, 384)

  Finding optimal k for praises (silhouette analysis)...
    k=4: silhouette=0.0719
    k=5: silhouette=0.0760
    k=6: silhouette=0.0837
    k=7: silhouette=0.0897
    k=8: silhouette=0.0925
    k=9: silhouette=0.0952
    k=10: silhouette=0.0896
    k=11: silhouette=0.0927
  → Optimal k=9 (score=0.0952)

Clustering praises into 9 clusters...
Generating cluster labels...
  Cluster  0 ( 220 praises) → Food Good Great
    Examples: ['fas managed well, food and drinks excellent', 'vegan special meal was thoughtfully prepared', 'outstanding cabin crew, comfortable aircraft, very good food and drink']
  Cluster  1 ( 103 praises) → Doha Best Lounge
    Examples: ['really impressed with their ground staff in doha', 'qatar staff did to help us', 'alsafwa lounge at doha is the best lounge i have seen.']
  Cluster  2 ( 108 praises) → Friendly Helpful Food
    Examples: ['exceptionally good staff, felt safe and well cared for', 'staff are always friendly and very 